# Notebook 1: Powerspectrum & AveragedPowerspectrum Framework


---

## Overview

This notebook demonstrates the basic framework for computing **power spectral density (PSD)** from X-ray light curves using the [Stingray](https://docs.stingray.science/) library.

We cover:
1. Simulating a light curve with a known power-law spectrum using `stingray.simulator`
2. Computing `Powerspectrum` with **Leahy**, **fractional rms**, and **absolute rms** normalizations
3. Computing `AveragedPowerspectrum` over multiple segments
4. Logarithmic rebinning for publication-quality plots
5. Verification: mean Leahy power ≈ 2.0 for Poisson noise

### References
- van der Klis, M. (1989). "Fourier Techniques in X-Ray Timing." NATO ASI Series, 262, 27–69.
- Leahy, D. A., et al. (1983). "On searches for pulsed emission." ApJ, 266, 160.
- Huppenkothen, D., et al. (2019). "Stingray: A Modern Python Library for Spectral Timing." ApJ, 881, 39.
- Stingray Simulator Docs: https://docs.stingray.science/en/stable/simulator.html

## 1. Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

from stingray import Lightcurve
from stingray.powerspectrum import Powerspectrum, AveragedPowerspectrum
from stingray.simulator import simulator

# Plot styling
rcParams['figure.figsize'] = (12, 6)
rcParams['font.size'] = 13
rcParams['axes.grid'] = True
rcParams['grid.alpha'] = 0.3

print("Imports successful!")

## 2. Simulate a Light Curve with Known Power Spectrum

We use `stingray.simulator.Simulator` to generate a synthetic light curve whose underlying power spectrum follows a **power-law** shape:

$$P(f) \propto f^{-\beta}$$

With $\beta = 2$ (random-walk / red noise), we know the true spectral shape, so we can verify that our power spectrum recovery is correct.

In [ ]:
# Simulator parameters
N = 65536       # Number of time bins (2^16 for efficient FFT)
dt = 1/256      # Time resolution: 1/256 s (like NICER)
mean_rate = 300  # Mean count rate (counts/s), typical for bright X-ray source
rms = 0.3        # Fractional rms variability (30%)

# Create simulator
sim = simulator.Simulator(N=N, mean=mean_rate, dt=dt, rms=rms)

# Simulate with power-law index beta=2 (random walk)
lc = sim.simulate(2)

print(f"Light curve duration: {lc.tseg:.1f} s")
print(f"Number of bins: {lc.n}")
print(f"Time resolution: {lc.dt} s")
print(f"Mean count rate: {np.mean(lc.counts):.1f} counts/s")

In [ ]:
# Plot the simulated light curve
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(lc.time, lc.counts, lw=0.5, color='#2196F3', alpha=0.8)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Count Rate (counts/s)')
ax.set_title('Simulated X-ray Light Curve (β=2, Red Noise)', fontsize=15, fontweight='bold')
ax.axhline(np.mean(lc.counts), color='#FF5722', ls='--', lw=1.5, label=f'Mean = {np.mean(lc.counts):.0f} cts/s')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Computing the Powerspectrum

### Normalization Options

Stingray supports several normalizations (van der Klis 1989):

| Normalization | Formula | Units | Use Case |
|:---|:---|:---|:---|
| **Leahy** | $P_j = 2\|a_j\|^2 / N_{\text{phot}}$ | dimensionless | Compare with $\chi^2_2$ distribution |
| **fractional rms** | $P_j = 2\|a_j\|^2 / (\bar{x}^2 T)$ | Hz$^{-1}$ | Source-intrinsic variability |
| **absolute rms** | $P_j = 2\|a_j\|^2 / T$ | (counts/s)² Hz$^{-1}$ | Absolute variability amplitude |

In [ ]:
# Compute Powerspectrum with Leahy normalization
ps_leahy = Powerspectrum(lc, norm='leahy')

print(f"Frequency range: {ps_leahy.freq[0]:.4f} — {ps_leahy.freq[-1]:.1f} Hz")
print(f"Frequency resolution: {ps_leahy.df:.6f} Hz")
print(f"Number of frequency bins: {len(ps_leahy.freq)}")
print(f"Mean Leahy power: {np.mean(ps_leahy.power):.4f} (expected ≈ 2.0 for Poisson noise)")

In [ ]:
# Compute with different normalizations
ps_rms = Powerspectrum(lc, norm='frac')
ps_abs = Powerspectrum(lc, norm='abs')

# Plot all three normalizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, ps, label, color in zip(axes, 
    [ps_leahy, ps_rms, ps_abs],
    ['Leahy', 'Fractional RMS', 'Absolute RMS'],
    ['#4CAF50', '#FF9800', '#9C27B0']):
    
    ax.loglog(ps.freq, ps.power, lw=0.3, alpha=0.5, color=color)
    # Rebin for clarity
    ps_rb = ps.rebin_log(f=0.03)
    ax.loglog(ps_rb.freq, ps_rb.power, 'o-', ms=3, lw=1.5, color=color)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Power')
    ax.set_title(f'{label} Normalization', fontsize=13, fontweight='bold')

plt.suptitle('Power Spectrum — Three Normalizations', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. AveragedPowerspectrum

For real data, we split the light curve into **multiple segments** and average the power spectra to reduce noise (Leahy et al. 1983). The variance of the averaged spectrum decreases as $1/M$ where $M$ is the number of segments.

This is the standard approach for X-ray timing with NICER, RXTE, XMM-Newton, etc.

In [ ]:
# Compute AveragedPowerspectrum with 16-second segments
seg_size = 16.0  # seconds per segment

avg_ps = AveragedPowerspectrum(lc, segment_size=seg_size, norm='leahy')

n_segments = int(lc.tseg / seg_size)
print(f"Segment size: {seg_size} s")
print(f"Number of segments: {n_segments}")
print(f"Frequency resolution: {avg_ps.df:.4f} Hz")
print(f"Nyquist frequency: {avg_ps.freq[-1]:.1f} Hz")
print(f"Mean Leahy power: {np.mean(avg_ps.power):.4f}")

In [ ]:
# Compare single vs averaged power spectrum
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Single PDS
ps_rb = ps_leahy.rebin_log(f=0.03)
axes[0].loglog(ps_leahy.freq, ps_leahy.power, lw=0.2, alpha=0.3, color='gray')
axes[0].loglog(ps_rb.freq, ps_rb.power, 'o-', ms=3, color='#2196F3', label='Log-rebinned')
axes[0].axhline(2.0, color='#F44336', ls='--', lw=1.5, label='Poisson level = 2')
axes[0].set_title('Single Powerspectrum', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Leahy Power')
axes[0].legend()

# Averaged PDS
avg_rb = avg_ps.rebin_log(f=0.03)
axes[1].loglog(avg_ps.freq, avg_ps.power, lw=0.2, alpha=0.3, color='gray')
axes[1].loglog(avg_rb.freq, avg_rb.power, 'o-', ms=3, color='#FF9800', label=f'Averaged ({n_segments} segs)')
axes[1].axhline(2.0, color='#F44336', ls='--', lw=1.5, label='Poisson level = 2')
axes[1].set_title(f'AveragedPowerspectrum ({n_segments} segments)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Leahy Power')
axes[1].legend()

plt.suptitle('Single vs. Averaged Power Spectrum (Leahy Normalization)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Logarithmic Rebinning

At high frequencies, the power spectrum has many closely-spaced bins with similar power values. **Logarithmic rebinning** groups these bins geometrically, improving signal-to-noise while preserving the spectral shape.

The rebinning factor $f$ means each successive bin is $(1+f)$ times wider than the previous one.

In [ ]:
# Demonstrate different rebinning factors
fig, ax = plt.subplots(figsize=(12, 6))

# Raw spectrum (gray background)
ax.loglog(avg_ps.freq, avg_ps.power, lw=0.2, alpha=0.2, color='gray', label='Raw')

# Different rebin factors
for f, color, marker in [(0.01, '#4CAF50', 's'), (0.03, '#2196F3', 'o'), (0.1, '#FF5722', 'D')]:
    rb = avg_ps.rebin_log(f=f)
    ax.loglog(rb.freq, rb.power, f'{marker}-', ms=4, lw=1.2, color=color, 
              label=f'f = {f} ({len(rb.freq)} bins)')

ax.axhline(2.0, color='#F44336', ls=':', lw=1, alpha=0.7)
ax.set_xlabel('Frequency (Hz)', fontsize=13)
ax.set_ylabel('Leahy Power', fontsize=13)
ax.set_title('Effect of Logarithmic Rebinning', fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 6. Verification: Poisson Noise Level

For **Leahy normalization**, the expected power from pure Poisson noise is exactly **2.0**. This is a fundamental property we can verify:

$$\langle P_{\text{Leahy}} \rangle = 2$$

We simulate a **pure noise** light curve (no red noise) and check.

In [ ]:
# Create a pure Poisson noise light curve
np.random.seed(42)
times = np.arange(0, 256, dt)
counts = np.random.poisson(mean_rate * dt, size=len(times)) / dt  # Poisson counts → rate
lc_noise = Lightcurve(times, counts, dt=dt)

# Compute power spectrum
ps_noise = Powerspectrum(lc_noise, norm='leahy')

mean_power = np.mean(ps_noise.power)
print(f"Mean Leahy power (Poisson noise): {mean_power:.4f}")
print(f"Expected: 2.0000")
print(f"Deviation: {abs(mean_power - 2.0):.4f} ({abs(mean_power - 2.0)/2.0*100:.2f}%)")
print(f"\n✅ Verification {'PASSED' if abs(mean_power - 2.0) < 0.1 else 'FAILED'}!")

## 7. Julia (Stingray.jl) API Preview

The GSoC 2026 project will implement these same operations in Julia. The planned API:

```julia
using Stingray

# Read events and create light curve
evt = EventList("observation.fits")
lc = LightCurve(evt, dt=1/256)

# Powerspectrum with different normalizations
ps = Powerspectrum(lc, norm=:leahy)
ps_rms = Powerspectrum(lc, norm=:frac)

# Averaged over segments (handles GTIs automatically)
avg_ps = AveragedPowerspectrum(lc, segment_size=16.0, norm=:leahy)

# Logarithmic rebinning
ps_rb = rebin_log(avg_ps, f=0.03)
```